In [ ]:
!pip install yfinance pandas numpy fredapi scikit-learn

In [13]:
# =====================================================================
#  GLOBAL AI MOMENTUM INDICATOR (GAMI)
#  Modelo de momentum IA vs SaaS con coste de deuda y riesgo de burbuja
# =====================================================================

import yfinance as yf
import pandas as pd
import numpy as np
from fredapi import Fred
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------
#  CONFIGURACIÓN
# ---------------------------------------------------------------------
FRED_API_KEY = "67f32747fea7584414ad3688f8d9c99a"   # <-- Reemplaza con tu clave

AI_TICKERS = ["NVDA", "AMD", "AVGO", "MU", "TSM", "MSFT", "GOOGL", "AMZN", "META", "PLTR"]
SAAS_TICKERS = ["CRM", "NOW", "WDAY", "SNOW", "DDOG", "MDB", "TEAM", "OKTA", "CRWD", "ZS"]
BENCHMARK = "SPY"

SHORT_WINDOW  = 21     # ~1 mes
MEDIUM_WINDOW = 63     # ~3 meses
LONG_WINDOW   = 126    # ~6 meses


# ---------------------------------------------------------------------
#  UTILIDADES
# ---------------------------------------------------------------------
def _as_series(x, index):
    """Convierte cualquier entrada (DataFrame, array, escalar) en Series alineada."""
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]                      # primera columna si es 2D
    if not isinstance(x, pd.Series):
        x = pd.Series(x, index=index)
    return x.reindex(index).astype(float)


# ---------------------------------------------------------------------
#  1. DESCARGA DE DATOS
# ---------------------------------------------------------------------
def fetch_stock_data(tickers, start_date, end_date):
    """
    Descarga precios y volúmenes y aplana el MultiIndex a columnas planas:
        TICKER_Close, TICKER_Volume, ...
    """
    data = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        group_by="ticker",
        auto_adjust=True,
        progress=False,
        threads=True,
    )
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [f"{t}_{f}" for t, f in data.columns]
    return data


def fetch_fred_data(api_key, series_ids, start_date, end_date):
    """Descarga series macro de FRED."""
    fred = Fred(api_key=api_key)
    df = pd.DataFrame()
    for sid in series_ids:
        try:
            s = fred.get_series(sid,
                                observation_start=start_date,
                                observation_end=end_date)
            df[sid] = s
        except Exception as e:
            print(f"[FRED] No se pudo obtener {sid}: {e}")
    df.index = pd.to_datetime(df.index)
    return df


# ---------------------------------------------------------------------
#  2. MOMENTUM POR CESTA
# ---------------------------------------------------------------------
def compute_basket_momentum(ticker_data, tickers, benchmark_data, benchmark,
                            short_win=SHORT_WINDOW,
                            medium_win=MEDIUM_WINDOW,
                            long_win=LONG_WINDOW):
    """Calcula métricas de momentum para una cesta. Devuelve Series 1D."""
    # Precios de cierre
    basket_close = pd.DataFrame(index=ticker_data.index)
    for t in tickers:
        col = f"{t}_Close"
        if col in ticker_data.columns:
            basket_close[t] = pd.to_numeric(ticker_data[col], errors="coerce").squeeze()

    if basket_close.empty:
        raise ValueError("No se extrajo ningún precio. Revisa tickers o yfinance.")

    # Índice equiponderado normalizado a 100
    basket_norm  = basket_close.div(basket_close.iloc[0]).mul(100)
    basket_index = basket_norm.mean(axis=1).squeeze()

    # Benchmark
    bench_col = f"{benchmark}_Close"
    if bench_col in benchmark_data.columns:
        bench_close = pd.to_numeric(benchmark_data[bench_col], errors="coerce").squeeze()
    elif "Close" in benchmark_data.columns:
        bench_close = pd.to_numeric(benchmark_data["Close"], errors="coerce").squeeze()
    else:
        raise ValueError(f"No se encontró la columna de cierre de {benchmark}.")

    bench_close  = bench_close.reindex(basket_index.index).ffill()
    basket_index = basket_index.squeeze()

    # Momentum
    roc_short  = basket_index.pct_change(short_win).squeeze()
    roc_medium = basket_index.pct_change(medium_win).squeeze()
    roc_long   = basket_index.pct_change(long_win).squeeze()

    bench_roc_medium = bench_close.pct_change(medium_win).squeeze()
    rel_strength     = (roc_medium - bench_roc_medium).squeeze()

    sma_50       = basket_index.rolling(50).mean().squeeze()
    trend_signal = (basket_index > sma_50).astype(int).squeeze()

    return {
        "basket_index": basket_index,
        "roc_short":    roc_short,
        "roc_medium":   roc_medium,
        "roc_long":     roc_long,
        "rel_strength": rel_strength,
        "trend_signal": trend_signal,
        "sma_50":       sma_50,
    }


# ---------------------------------------------------------------------
#  3. COSTE DE DEUDA
# ---------------------------------------------------------------------
def compute_debt_stress(fred_df, corporate_series="BAMLC0A0CM", treasury_series="DGS10"):
    """Calcula spread corporativo y su momentum."""
    df = fred_df.ffill()
    spread          = (df[corporate_series] - df[treasury_series]).squeeze()
    spread_momentum = spread.diff(63).squeeze()
    spread_mean_1y  = spread.rolling(252).mean().squeeze()
    debt_stress     = (spread > spread_mean_1y).astype(int).squeeze()

    return pd.DataFrame({
        "credit_spread":   spread,
        "spread_momentum": spread_momentum,
        "debt_stress":     debt_stress,
    })


# ---------------------------------------------------------------------
#  4. RIESGO DE BURBUJA
# ---------------------------------------------------------------------
def compute_bubble_risk(ai_momentum_dict, ticker_data, ai_tickers):
    """Proxy simplificado del riesgo de burbuja IA."""
    basket_idx = ai_momentum_dict["basket_index"].squeeze()
    roc_long   = ai_momentum_dict["roc_long"].squeeze()

    # 1) Desacoplamiento precio-momentum
    recent_return = basket_idx.pct_change(21).squeeze()
    decoupling    = (recent_return - roc_long).squeeze()

    # 2) Pico de volatilidad
    daily_returns = basket_idx.pct_change().squeeze()
    volatility_21 = (daily_returns.rolling(21).std() * np.sqrt(252)).squeeze()
    vol_mean_6m   = volatility_21.rolling(126).mean().squeeze()
    vol_spike     = (volatility_21 / vol_mean_6m).squeeze()

    # 3) Volumen agregado
    volume_series = pd.DataFrame(index=ticker_data.index)
    for t in ai_tickers:
        col = f"{t}_Volume"
        if col in ticker_data.columns:
            volume_series[t] = pd.to_numeric(ticker_data[col], errors="coerce").squeeze()

    total_volume = volume_series.sum(axis=1).squeeze()
    volume_ratio = (total_volume / total_volume.rolling(126).mean()).squeeze()

    bubble_df = pd.DataFrame({
        "decoupling":    decoupling,
        "volatility_21": volatility_21,
        "vol_spike":     vol_spike,
        "volume_ratio":  volume_ratio,
    })

    components = bubble_df[["decoupling", "vol_spike", "volume_ratio"]].dropna()
    if len(components) > 20:
        scaler = StandardScaler()
        z = scaler.fit_transform(components)
        bubble_df.loc[components.index, "bubble_risk_composite"] = z.mean(axis=1)

    return bubble_df


# ---------------------------------------------------------------------
#  5. INDICADOR GLOBAL
# ---------------------------------------------------------------------
def compute_global_ai_momentum(ai_mom, saas_mom, debt_stress, bubble_risk,
                               weights=None, rolling_z=252):
    if weights is None:
        weights = {"ai_momentum": 0.25, "saas_momentum": 0.25,
                   "debt_cost":   0.25, "bubble_risk":   0.25}

    idx = pd.DatetimeIndex(ai_mom["roc_medium"].index)
    df  = pd.DataFrame(index=idx)

    # Pilar 1: IA
    ai_roc = _as_series(ai_mom["roc_medium"],   idx).fillna(0)
    ai_rs  = _as_series(ai_mom["rel_strength"], idx).fillna(0)
    df["ai_momentum_raw"] = (ai_roc + ai_rs) / 2

    # Pilar 2: SaaS
    saas_roc = _as_series(saas_mom["roc_medium"],   idx).fillna(0)
    saas_tr  = _as_series(saas_mom["trend_signal"], idx).fillna(0)
    df["saas_momentum_raw"] = (saas_roc + saas_tr) / 2

    # Pilar 3: Coste de deuda (invertido)
    spread_mom = _as_series(debt_stress["spread_momentum"], idx).fillna(0)
    df["debt_cost_raw"] = -spread_mom

    # Pilar 4: Riesgo de burbuja (invertido)
    if "bubble_risk_composite" in bubble_risk.columns:
        br = _as_series(bubble_risk["bubble_risk_composite"], idx).fillna(0)
    else:
        br = pd.Series(0.0, index=idx)
    df["bubble_risk_raw"] = -br

    # Z-score rodante
    for col in ["ai_momentum_raw", "saas_momentum_raw",
                "debt_cost_raw",   "bubble_risk_raw"]:
        mean = df[col].rolling(rolling_z, min_periods=20).mean()
        std  = df[col].rolling(rolling_z, min_periods=20).std().replace(0, np.nan)
        df[col.replace("_raw", "")] = ((df[col] - mean) / std).squeeze()

    # GAMI compuesto
    df["GAMI"] = (
        weights["ai_momentum"]   * df["ai_momentum"].fillna(0) +
        weights["saas_momentum"] * df["saas_momentum"].fillna(0) +
        weights["debt_cost"]     * df["debt_cost"].fillna(0) +
        weights["bubble_risk"]   * df["bubble_risk"].fillna(0)
    )
    return df


# ---------------------------------------------------------------------
#  6. PIPELINE
# ---------------------------------------------------------------------
def run_model(start_date=None, end_date=None):
    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")
    if start_date is None:
        start_date = (datetime.today() - timedelta(days=3*365)).strftime("%Y-%m-%d")

    # Descarga conjunta (IA + SaaS + SPY)
    all_tickers = list(set(AI_TICKERS + SAAS_TICKERS + [BENCHMARK]))
    ticker_data = fetch_stock_data(all_tickers, start_date, end_date)

    # Momentum de cestas (benchmark del mismo DataFrame aplanado)
    ai_mom   = compute_basket_momentum(ticker_data, AI_TICKERS,   ticker_data, BENCHMARK)
    saas_mom = compute_basket_momentum(ticker_data, SAAS_TICKERS, ticker_data, BENCHMARK)

    # Deuda
    fred_df = fetch_fred_data(FRED_API_KEY,
                              ["BAMLC0A0CM", "BAMLH0A0HYM2", "DGS10"],
                              start_date, end_date)
    debt_stress = compute_debt_stress(fred_df)

    # Burbuja
    bubble_risk = compute_bubble_risk(ai_mom, ticker_data, AI_TICKERS)

    # GAMI
    gami_df = compute_global_ai_momentum(ai_mom, saas_mom, debt_stress, bubble_risk)

    # Output
    valid = gami_df.dropna()
    if valid.empty:
        print("⚠️  No hay datos suficientes para calcular el GAMI.")
        return gami_df, ai_mom, saas_mom, debt_stress, bubble_risk

    latest = valid.iloc[-1]
    print("\n=== GLOBAL AI MOMENTUM INDICATOR — ÚLTIMA LECTURA ===")
    print(f"Fecha            : {valid.index[-1].date()}")
    print(f"GAMI             : {latest['GAMI']:+.3f}")
    print(f"  Momentum IA    : {latest['ai_momentum']:+.3f}")
    print(f"  Momentum SaaS  : {latest['saas_momentum']:+.3f}")
    print(f"  Coste deuda    : {latest['debt_cost']:+.3f}")
    print(f"  Riesgo burbuja : {latest['bubble_risk']:+.3f}")

    return gami_df, ai_mom, saas_mom, debt_stress, bubble_risk


# ---------------------------------------------------------------------
#  EJECUCIÓN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    gami_df, ai_mom, saas_mom, debt_stress, bubble_risk = run_model()


=== GLOBAL AI MOMENTUM INDICATOR — ÚLTIMA LECTURA ===
Fecha            : 2026-09-21
GAMI             : +0.344
  Momentum IA    : -0.566
  Momentum SaaS  : +1.108
  Coste deuda    : -0.409
  Riesgo burbuja : +1.242
